#### Défi quotidien : Analyse stratégique des performances des supermarchés

En tant que **Data Scientist** senior, vous êtes chargé d'analyser un dataset de ventes de supermarchés (Superstore) dans le but de fournir aux décideurs des informations exploitables. Vos objectifs sont multiples :

1. **Comprendre** la structure du jeu de données, le contenu des variables et la qualité des données.

2. **Nettoyer** et prétraiter les données pour les rendre adaptées à l'analyse.

3. **Analyser** les performances des ventes via des visualisations interactives et statiques.

4. **Identifier** les produits, catégories et régions les plus rentables.

5. **Fournir** un rapport exécutif synthétisant vos constats et recommandations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown, IntSlider
import plotly.express as px
import plotly.graph_objects as go
import time
import warnings
warnings.filterwarnings('ignore')

#### Nettoyez et prétraitez vos données :

In [ ]:
df = pd.read_csv('dataset/Sample - Superstore.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print('\n--- Info ---')
df.info()
print('\n--- Describe ---')
print(df.describe())
print('\n--- Missing Values ---')
print(df.isnull().sum())

#### Corriger les types de données

In [ ]:
print(f'Duplicate rows: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'After removal - Shape: {df.shape}')
df['Postal Code'] = df['Postal Code'].fillna(0)
print(f'Postal Code NaN filled with 0')
print(f'Missing values after fill:\n{df.isnull().sum()}')

#### Ingénierie des fonctionnalités

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')
print(f'Order Date dtype: {df["Order Date"].dtype}')
print(f'Ship Date dtype: {df["Ship Date"].dtype}')
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')
print(f'New columns created: Profit Margin, Order Year, Order Month, Order Month-Year')

**2. Analyse exploratoire approfondie (Matplotlib)**

In [ ]:
def plot_monthly_sales(category):
    data = df[df['Category'] == category]
    monthly = data.groupby('Order Month-Year')['Sales'].sum()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(monthly.index.astype(str), monthly.values, marker='o')
    ax.set_title(f'Monthly Sales for {category}')
    ax.set_xlabel('Month-Year')
    ax.set_ylabel('Sales')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

categories = df['Category'].unique().tolist()
interact(plot_monthly_sales, category=Dropdown(options=categories, description='Category:'))

**Performances des ventes géographiques**

In [ ]:
def plot_top_states(n):
    top_states = df.groupby('State')['Sales'].sum().nlargest(n)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(top_states.index, top_states.values)
    ax.set_title(f'Top {n} States by Sales')
    ax.set_xlabel('Sales')
    plt.tight_layout()
    plt.show()

interact(plot_top_states, n=IntSlider(min=1, max=20, value=5, description='Top N:'))

**3. Communiquer des informations (Seaborn)**

In [ ]:
top_products = df.groupby('Product Name')['Profit'].sum().nlargest(10)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_products.values, y=top_products.index, ax=ax)
ax.set_title('Top 10 Profitable Products')
ax.set_xlabel('Profit')
for i, v in enumerate(top_products.values):
    ax.text(v, i, f' ${v:,.0f}', va='center')
plt.tight_layout()
plt.show()
print('Executive Summary: Top 10 products drive significant profit. Consider increasing inventory for these items.')

**Diagramme de dispersion Remise vs Profit**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='Discount', y='Profit', ax=ax)
sns.regplot(data=df, x='Discount', y='Profit', ax=ax, scatter=False, color='red')
ax.set_title('Discount vs Profit')
ax.set_xlabel('Discount')
ax.set_ylabel('Profit')
plt.tight_layout()
plt.show()
print('Insights: Higher discounts correlate with lower profits in some cases. Review discount strategy.')

**4. Revue de la méthodologie et des outils**

In [ ]:
print('Library Comparison: Pandas for data manipulation, Matplotlib/Seaborn for static visualizations, Plotly for interactive charts.')
start = time.time()
_ = df.groupby('Category')['Sales'].sum()
end = time.time()
print(f'Pandas groupby timing: {end - start:.4f} seconds')

**5. Livrable final**

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
avg_margin = df['Profit Margin'].mean()
print(f'Total Sales: ${total_sales:,.2f}')
print(f'Total Profit: ${total_profit:,.2f}')
print(f'Average Profit Margin: {avg_margin:.2f}%')
print('Recommendation: Focus on top-performing products and optimize discount policies.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
df.groupby('Order Year')['Sales'].sum().plot(ax=axes[0, 0], kind='bar')
axes[0, 0].set_title('Sales by Year')
axes[0, 0].set_ylabel('Sales')
df.groupby('Region')['Profit'].sum().plot(ax=axes[0, 1], kind='bar')
axes[0, 1].set_title('Profit by Region')
axes[0, 1].set_ylabel('Profit')
df['Profit Margin'].hist(ax=axes[1, 0], bins=30)
axes[1, 0].set_title('Profit Margin Distribution')
axes[1, 0].set_xlabel('Profit Margin')
df.groupby('Category')['Discount'].mean().plot(ax=axes[1, 1], kind='bar')
axes[1, 1].set_title('Avg Discount by Category')
axes[1, 1].set_ylabel('Discount')
plt.tight_layout()
plt.show()

*Annoter les valeurs aberrantes dans le tableau Remise vs. Profit*

In [ ]:
top3 = df.nlargest(3, 'Profit')[['Discount', 'Profit', 'Product Name']]
bottom3 = df.nsmallest(3, 'Profit')[['Discount', 'Profit', 'Product Name']]
print('Top 3 Profit outliers:')
print(top3)
print('\nBottom 3 Profit outliers:')
print(bottom3)

*Recréez un graphique interactif avec Plotly Express*

In [ ]:
fig = px.scatter(df, x='Discount', y='Profit', hover_data=['Product Name', 'Category'], trendline='ols')
fig.update_layout(title='Interactive Discount vs Profit Scatter', xaxis_title='Discount', yaxis_title='Profit')
fig.show()

#### Reponse

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
import time
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('dataset/Sample - Superstore.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

df.info()
df.describe()
df.isnull().sum()

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

print("\nMissing values per column:")
print(df.isnull().sum())

if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

date_columns = ['Order Date', 'Ship Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%d-%m-%Y')

print("\nData types after conversion:")
print(df[date_columns].dtypes)

df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print("\nNew features created:")
print(df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head())

monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    plt.figure(figsize=(12, 6))
    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        plt.plot(total_monthly.index.to_timestamp(), total_monthly.values, 
                marker='o', linewidth=2, markersize=4, color='steelblue')
        plt.title('Monthly Sales Trend - All Categories', fontsize=16, fontweight='bold')
    else:
        category_data = monthly_sales[monthly_sales['Category'] == category]
        plt.plot(category_data['Date'], category_data['Sales'], 
                marker='o', linewidth=2, markersize=4, color='steelblue')
        plt.title(f'Monthly Sales Trend - {category}', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Sales ($)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

categories = ['All'] + list(df['Category'].unique())
category_dropdown = Dropdown(options=categories, value='All', description='Category:')
interact(plot_monthly_sales, category=category_dropdown)

state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    plt.figure(figsize=(12, max(6, top_n * 0.4)))
    top_states = state_sales.tail(top_n)
    bars = plt.barh(range(len(top_states)), top_states.values, color='steelblue')
    plt.yticks(range(len(top_states)), top_states.index)
    plt.xlabel('Total Sales ($)', fontsize=12)
    plt.ylabel('State', fontsize=12)
    plt.title(f'Top {top_n} States by Sales Performance', fontsize=16, fontweight='bold')
    for i, (state, value) in enumerate(top_states.items()):
        plt.text(value + max(top_states.values()) * 0.01, i, f'${value:,.0f}', 
                va='center', fontsize=10)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Total states analyzed: {len(state_sales)}")
    print(f"Top {top_n} states represent: ${top_states.sum():,.0f} in sales")

top_n_slider = IntSlider(min=5, max=25, value=10, description='Top N States:')
interact(plot_top_states, top_n=top_n_slider)

product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 8))
ax = sns.barplot(x=product_profit.values, y=product_profit.index, 
                palette='viridis', orient='h')
plt.title('Top 10 Most Profitable Products\nExecutive Summary - Product Performance Analysis', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Total Profit ($)', fontsize=12, fontweight='bold')
plt.ylabel('Product Name', fontsize=12, fontweight='bold')
for i, (product, profit) in enumerate(product_profit.items()):
    ax.text(profit + max(product_profit.values()) * 0.01, i, f'${profit:,.0f}', 
            va='center', fontweight='bold', fontsize=10)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Key Insights:")
print(f"• Most profitable product generates: ${product_profit.iloc[0]:,.0f}")
print(f"• Top 10 products contribute: ${product_profit.sum():,.0f} total profit")
print(f"• Average profit per top product: ${product_profit.mean():,.0f}")

plt.figure(figsize=(14, 8))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', 
               alpha=0.6, s=50)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, 
           color='red', line_kws={'linewidth': 2, 'linestyle': '--'})
plt.title('Discount Strategy Analysis: Impact on Profitability by Category', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Discount Rate', fontsize=12, fontweight='bold')
plt.ylabel('Profit ($)', fontsize=12, fontweight='bold')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
plt.text(0.5, 50, 'Break-even line', fontsize=10, alpha=0.7)
plt.grid(True, alpha=0.3)
plt.legend(title='Product Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("Discount Analysis Insights:")
high_discount = df[df['Discount'] > 0.2]
print(f"• Transactions with >20% discount: {len(high_discount):,}")
print(f"• Average profit for high discounts: ${high_discount['Profit'].mean():.2f}")
print(f"• Percentage of high-discount sales with losses: {(high_discount['Profit'] < 0).mean()*100:.1f}%")

print("\nCategory-specific discount impact:")
for category in df['Category'].unique():
    cat_data = df[df['Category'] == category]
    high_disc_cat = cat_data[cat_data['Discount'] > 0.2]
    if len(high_disc_cat) > 0:
        avg_loss = high_disc_cat['Profit'].mean()
        print(f"• {category}: Average profit at >20% discount = ${avg_loss:.2f}")

print("=== LIBRARY COMPARISON ANALYSIS ===")
print("\nMATPLOTLIB STRENGTHS (from our analysis):")
print("• Fine-grained control over interactive widgets")
print("• Custom annotations and text positioning") 
print("• Precise subplot layouts and figure sizing")
print("• Integration with ipywidgets for dynamic updates")
print("\nSEABORN STRENGTHS (from our analysis):")
print("• Built-in statistical visualizations (regplot)")
print("• Automatic color palettes and legends")
print("• Clean, publication-ready default styling")
print("• Easy categorical data visualization")
print("\nSPEED COMPARISON:")
start = time.time()
plt.figure(figsize=(8, 6))
plt.plot(df.groupby('Order Year')['Sales'].sum())
plt.close()
matplotlib_time = time.time() - start

start = time.time()
plt.figure(figsize=(8, 6))
sns.lineplot(data=df.groupby('Order Year')['Sales'].sum().reset_index(), 
             x='Order Year', y='Sales')
plt.close()
seaborn_time = time.time() - start

print(f"• Matplotlib basic plot: {matplotlib_time:.4f} seconds")
print(f"• Seaborn equivalent: {seaborn_time:.4f} seconds")

print("\n=== EXECUTIVE SUMMARY - KEY FINDINGS ===\n")
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
profit_margin = (total_profit / total_sales) * 100

print("📊 BUSINESS PERFORMANCE:")
print(f"• Total Revenue: ${total_sales:,.0f}")
print(f"• Total Profit: ${total_profit:,.0f}")
print(f"• Overall Profit Margin: {profit_margin:.1f}%")
print()

top_state = state_sales.index[-1]
top_state_sales = state_sales.iloc[-1]
print("🗺️ GEOGRAPHIC PERFORMANCE:")
print(f"• Top performing state: {top_state} (${top_state_sales:,.0f})")
print(f"• Geographic concentration: Top 5 states = {(state_sales.tail(5).sum()/total_sales)*100:.1f}% of sales")
print()

top_category = df.groupby('Category')['Sales'].sum().sort_values(ascending=False).index[0]
print("🏆 PRODUCT PERFORMANCE:")
print(f"• Leading category: {top_category}")
print(f"• Most profitable product: {product_profit.index[0]}")
print()

high_discount_loss_rate = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100
print("💰 DISCOUNT STRATEGY INSIGHTS:")
print(f"• High discount risk: {high_discount_loss_rate:.1f}% of >20% discounts result in losses")
print(f"• Recommended max discount threshold: 20% to maintain profitability")

def create_dashboard():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
    ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values, marker='o')
    ax1.set_title('Monthly Sales Trend')
    ax1.tick_params(axis='x', rotation=45)
    category_sales = df.groupby('Category')['Sales'].sum()
    ax2.bar(category_sales.index, category_sales.values)
    ax2.set_title('Sales by Category')
    top_10_states = state_sales.tail(10)
    ax3.barh(range(len(top_10_states)), top_10_states.values)
    ax3.set_yticks(range(len(top_10_states)))
    ax3.set_yticklabels(top_10_states.index)
    ax3.set_title('Top 10 States by Sales')
    for category in df['Category'].unique():
        cat_data = df[df['Category'] == category]
        ax4.scatter(cat_data['Discount'], cat_data['Profit'], 
                   label=category, alpha=0.6)
    ax4.set_xlabel('Discount')
    ax4.set_ylabel('Profit')
    ax4.set_title('Discount vs Profit by Category')
    ax4.legend()
    ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

create_dashboard()

plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=0.6)
top_3_profitable = df.nlargest(3, 'Profit')
bottom_3_profitable = df.nsmallest(3, 'Profit')
for idx, row in top_3_profitable.iterrows():
    plt.annotate(f'Best: ${row["Profit"]:.0f}', 
                xy=(row['Discount'], row['Profit']),
                xytext=(10, 10), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='green', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
for idx, row in bottom_3_profitable.iterrows():
    plt.annotate(f'Worst: ${row["Profit"]:.0f}', 
                xy=(row['Discount'], row['Profit']),
                xytext=(10, -20), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
plt.title('Discount vs Profit Analysis with Outlier Identification')
plt.show()

fig = px.scatter(df, x='Discount', y='Profit', color='Category',
                hover_data=['Product Name', 'Sales'], 
                title='Interactive Discount vs Profit Analysis (Plotly)')
fig.add_traces(px.scatter(df, x='Discount', y='Profit', trendline='ols').data[1])
fig.show()

print("PLOTLY vs MATPLOTLIB COMPARISON:")
print("Plotly Advantages:")
print("• Built-in interactivity (zoom, pan, hover)")
print("• Easy to share online")
print("• Professional tooltips and legends")
print("• Automatic responsive design")
print("\nMatplotlib + ipywidgets Advantages:") 
print("• More customization control")
print("• Better integration with Jupyter workflows")
print("• Smaller file sizes")
print("• Familiar to Python data scientists")
